In [0]:
# Paste your copied workspace file path here
path = "/Workspace/Users/azuser7416_mml.local@karthikirisoutlook.onmicrosoft.com/bookstore_orders_databricks.csv"

df = spark.read.csv(
    "file:" + path,
    header=True,
    inferSchema=True
)

display(df)

order_id,city,book_category,quantity,unit_price,order_status
1001,Hyderabad,Technology,2,850,Completed
1002,Bengaluru,Fiction,3,400,Completed
1003,Chennai,Science,1,650,Completed
1004,Hyderabad,Fiction,5,350,Completed
1005,Pune,History,2,500,Cancelled
1006,Bengaluru,Technology,1,1200,Completed
1007,Chennai,Fiction,4,300,Completed
1008,Pune,Science,2,750,Completed
1009,Hyderabad,History,3,550,Completed
1010,Bengaluru,Science,2,700,Cancelled


**Azure Databricks | DataFrame Practice**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
 
catalog = "sara_01"
schema  = "default"
volume  = "cinema_data"
 
csv_path  = "/Volumes/sara_01/default/cinema_data/cinema_bookings.csv"
json_path = "/Volumes/sara_01/default/cinema_data/cinema_master.json"

**A. Read and inspect the data**

1. Load cinema_bookings.csv into a Spark DataFrame with column names and inferred data types.

In [0]:
bookings_df = spark.read.csv(csv_path, header=True, inferSchema=True)

2. Load cinema_master.json into a second DataFrame. Display both DataFrames.

In [0]:
cinemas_df = spark.read.json(json_path)
display(bookings_df)
display(cinemas_df)

booking_id,cinema_id,movie_title,booking_date,show_date,tickets,price_per_ticket,discount_pct,booking_status,payment_method
B1001,C001,Starlight Express,2026-10-01,2026-10-03,2,250,10,Confirmed,UPI
B1002,C002,The Last Island,2026-10-01,2026-10-02,3,300,0,Confirmed,Card
B1003,C003,Code Red,2026-10-02,2026-10-04,1,220,5,Cancelled,UPI
B1004,C004,River of Dreams,2026-10-02,2026-10-05,4,180,0,Confirmed,Cash
B1005,C005,Starlight Express,2026-10-03,2026-10-05,2,320,15,Confirmed,Card
B1006,C006,The Last Island,2026-10-03,2026-10-04,5,210,null,Confirmed,UPI
B1007,C001,Code Red,2026-10-04,2026-10-06,3,250,0,Confirmed,UPI
B1008,C002,River of Dreams,2026-10-04,2026-10-06,2,300,10,Refunded,Card
B1009,C003,Starlight Express,2026-10-05,2026-10-07,4,220,5,Confirmed,UPI
B1010,C004,The Last Island,2026-10-05,2026-10-08,1,180,0,Confirmed,Cash


cinema_id,cinema_name,facilities,location,screen_count
C001,Galaxy Screens,"List(Parking, Food Court, Wheelchair Access)","List(Hyderabad, West)",5
C002,Silverlight Multiplex,"List(Parking, Food Court)","List(Bengaluru, North)",7
C003,Metro Movies,"List(Food Court, Wheelchair Access)","List(Mumbai, Central)",4
C004,Coastal Cinema,List(Parking),"List(Chennai, South)",3
C005,Capital Cineplex,"List(Parking, Food Court, Wheelchair Access)","List(Delhi, East)",8
C006,Orchid Screens,"List(Food Court, Wheelchair Access)","List(Pune, West)",4


3. Print the schema of both DataFrames. Identify the nested field and array field in the JSON.

In [0]:
bookings_df.printSchema()
cinemas_df.printSchema()

root
 |-- booking_id: string (nullable = true)
 |-- cinema_id: string (nullable = true)
 |-- movie_title: string (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- show_date: date (nullable = true)
 |-- tickets: integer (nullable = true)
 |-- price_per_ticket: integer (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- booking_status: string (nullable = true)
 |-- payment_method: string (nullable = true)

root
 |-- cinema_id: string (nullable = true)
 |-- cinema_name: string (nullable = true)
 |-- facilities: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- location: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- zone: string (nullable = true)
 |-- screen_count: long (nullable = true)



4. Display the first 10 booking records and count the total number of CSV rows.

In [0]:
display(bookings_df.limit(10))
print("Total rows:", bookings_df.count())

booking_id,cinema_id,movie_title,booking_date,show_date,tickets,price_per_ticket,discount_pct,booking_status,payment_method
B1001,C001,Starlight Express,2026-10-01,2026-10-03,2,250,10,Confirmed,UPI
B1002,C002,The Last Island,2026-10-01,2026-10-02,3,300,0,Confirmed,Card
B1003,C003,Code Red,2026-10-02,2026-10-04,1,220,5,Cancelled,UPI
B1004,C004,River of Dreams,2026-10-02,2026-10-05,4,180,0,Confirmed,Cash
B1005,C005,Starlight Express,2026-10-03,2026-10-05,2,320,15,Confirmed,Card
B1006,C006,The Last Island,2026-10-03,2026-10-04,5,210,null,Confirmed,UPI
B1007,C001,Code Red,2026-10-04,2026-10-06,3,250,0,Confirmed,UPI
B1008,C002,River of Dreams,2026-10-04,2026-10-06,2,300,10,Refunded,Card
B1009,C003,Starlight Express,2026-10-05,2026-10-07,4,220,5,Confirmed,UPI
B1010,C004,The Last Island,2026-10-05,2026-10-08,1,180,0,Confirmed,Cash


Total rows: 32


5. Select only booking_id, movie_title, tickets and booking_status.

In [0]:
bookings_df.select("booking_id", "movie_title", "tickets", "booking_status").show()

+----------+-----------------+-------+--------------+
|booking_id|      movie_title|tickets|booking_status|
+----------+-----------------+-------+--------------+
|     B1001|Starlight Express|      2|     Confirmed|
|     B1002|  The Last Island|      3|     Confirmed|
|     B1003|         Code Red|      1|     Cancelled|
|     B1004|  River of Dreams|      4|     Confirmed|
|     B1005|Starlight Express|      2|     Confirmed|
|     B1006|  The Last Island|      5|     Confirmed|
|     B1007|         Code Red|      3|     Confirmed|
|     B1008|  River of Dreams|      2|      Refunded|
|     B1009|Starlight Express|      4|     Confirmed|
|     B1010|  The Last Island|      1|     Confirmed|
|     B1011|         Code Red|      2|     Confirmed|
|     B1012|  River of Dreams|      3|     Cancelled|
|     B1013|  The Last Island|      5|     Confirmed|
|     B1014|Starlight Express|      2|     Confirmed|
|     B1015|  River of Dreams|      1|     Confirmed|
|     B1016|         Code Re

**B. Filter, clean and transform**

6. Filter bookings where booking_status is Confirmed.

In [0]:
confirmed_df = bookings_df.filter(F.col("booking_status") == "Confirmed")
confirmed_df.show()

+----------+---------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+
|booking_id|cinema_id|      movie_title|booking_date| show_date|tickets|price_per_ticket|discount_pct|booking_status|payment_method|
+----------+---------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+
|     B1001|     C001|Starlight Express|  2026-10-01|2026-10-03|      2|             250|          10|     Confirmed|           UPI|
|     B1002|     C002|  The Last Island|  2026-10-01|2026-10-02|      3|             300|           0|     Confirmed|          Card|
|     B1004|     C004|  River of Dreams|  2026-10-02|2026-10-05|      4|             180|           0|     Confirmed|          Cash|
|     B1005|     C005|Starlight Express|  2026-10-03|2026-10-05|      2|             320|          15|     Confirmed|          Card|
|     B1006|     C006|  The Last Island|  2026-10-03|2026-10-04|     

7. Find bookings with more than 3 tickets and a ticket price above 200.

In [0]:
bookings_df.filter((F.col("tickets") > 3) & (F.col("price_per_ticket") > 200)).show()

+----------+---------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+
|booking_id|cinema_id|      movie_title|booking_date| show_date|tickets|price_per_ticket|discount_pct|booking_status|payment_method|
+----------+---------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+
|     B1006|     C006|  The Last Island|  2026-10-03|2026-10-04|      5|             210|        NULL|     Confirmed|           UPI|
|     B1009|     C003|Starlight Express|  2026-10-05|2026-10-07|      4|             220|           5|     Confirmed|           UPI|
|     B1013|     C001|  The Last Island|  2026-10-07|2026-10-09|      5|             250|          10|     Confirmed|          Card|
|     B1017|     C005|  The Last Island|  2026-10-09|2026-10-11|      4|             320|          15|      Refunded|          Card|
|     B1023|     C005|  River of Dreams|  2026-10-12|2026-10-14|     

8. Count missing values in discount_pct and payment_method.

In [0]:
missing_df = bookings_df.select(
    F.count(F.when(F.col("discount_pct").isNull(), 1)).alias("missing_discount_pct"),
    F.count(F.when(F.col("payment_method").isNull(), 1)).alias("missing_payment_method"),
).show()

+--------------------+----------------------+
|missing_discount_pct|missing_payment_method|
+--------------------+----------------------+
|                   5|                     2|
+--------------------+----------------------+



9. Create a cleaned DataFrame that replaces missing discounts with 0 and missing payment methods with
Unknown.

In [0]:
clean_df = (
    bookings_df
    .withColumn("payment_method",
                F.when(F.trim(F.col("payment_method")) == "", None).otherwise(F.col("payment_method")))
    .fillna({"discount_pct": 0, "payment_method": "Unknown"})
)
clean_df.show()

+----------+---------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+
|booking_id|cinema_id|      movie_title|booking_date| show_date|tickets|price_per_ticket|discount_pct|booking_status|payment_method|
+----------+---------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+
|     B1001|     C001|Starlight Express|  2026-10-01|2026-10-03|      2|             250|          10|     Confirmed|           UPI|
|     B1002|     C002|  The Last Island|  2026-10-01|2026-10-02|      3|             300|           0|     Confirmed|          Card|
|     B1003|     C003|         Code Red|  2026-10-02|2026-10-04|      1|             220|           5|     Cancelled|           UPI|
|     B1004|     C004|  River of Dreams|  2026-10-02|2026-10-05|      4|             180|           0|     Confirmed|          Cash|
|     B1005|     C005|Starlight Express|  2026-10-03|2026-10-05|     

10. Add gross_amount = tickets multiplied by price_per_ticket.

In [0]:
clean_df = clean_df.withColumn("gross_amount", F.col("tickets") * F.col("price_per_ticket"))
clean_df.show()

+----------+---------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+------------+
|booking_id|cinema_id|      movie_title|booking_date| show_date|tickets|price_per_ticket|discount_pct|booking_status|payment_method|gross_amount|
+----------+---------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+------------+
|     B1001|     C001|Starlight Express|  2026-10-01|2026-10-03|      2|             250|          10|     Confirmed|           UPI|         500|
|     B1002|     C002|  The Last Island|  2026-10-01|2026-10-02|      3|             300|           0|     Confirmed|          Card|         900|
|     B1003|     C003|         Code Red|  2026-10-02|2026-10-04|      1|             220|           5|     Cancelled|           UPI|         220|
|     B1004|     C004|  River of Dreams|  2026-10-02|2026-10-05|      4|             180|           0|     Confirmed|       

11. Add net_amount after applying discount_pct. Keep the amount as a numeric column.

In [0]:
clean_df = clean_df.withColumn(
    "net_amount",
    F.round(F.col("gross_amount") * (1 - F.col("discount_pct") / 100), 2).cast("double")
)
clean_df.show()

+----------+---------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+------------+----------+
|booking_id|cinema_id|      movie_title|booking_date| show_date|tickets|price_per_ticket|discount_pct|booking_status|payment_method|gross_amount|net_amount|
+----------+---------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+------------+----------+
|     B1001|     C001|Starlight Express|  2026-10-01|2026-10-03|      2|             250|          10|     Confirmed|           UPI|         500|     450.0|
|     B1002|     C002|  The Last Island|  2026-10-01|2026-10-02|      3|             300|           0|     Confirmed|          Card|         900|     900.0|
|     B1003|     C003|         Code Red|  2026-10-02|2026-10-04|      1|             220|           5|     Cancelled|           UPI|         220|     209.0|
|     B1004|     C004|  River of Dreams|  2026-10-02|2026-

12. Add a booking_size column: Large for 4 or more tickets, otherwise Regular.

In [0]:
clean_df = clean_df.withColumn(
    "booking_size",
    F.when(F.col("tickets") >= 4, "Large").otherwise("Regular")
)
clean_df.show()

+----------+---------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+------------+----------+------------+
|booking_id|cinema_id|      movie_title|booking_date| show_date|tickets|price_per_ticket|discount_pct|booking_status|payment_method|gross_amount|net_amount|booking_size|
+----------+---------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+------------+----------+------------+
|     B1001|     C001|Starlight Express|  2026-10-01|2026-10-03|      2|             250|          10|     Confirmed|           UPI|         500|     450.0|     Regular|
|     B1002|     C002|  The Last Island|  2026-10-01|2026-10-02|      3|             300|           0|     Confirmed|          Card|         900|     900.0|     Regular|
|     B1003|     C003|         Code Red|  2026-10-02|2026-10-04|      1|             220|           5|     Cancelled|           UPI|         220|     

13. Sort bookings by net_amount from highest to lowest and show the top 5.

In [0]:
clean_df.orderBy(F.col("net_amount").desc()).limit(5).show()

+----------+---------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+------------+----------+------------+
|booking_id|cinema_id|      movie_title|booking_date| show_date|tickets|price_per_ticket|discount_pct|booking_status|payment_method|gross_amount|net_amount|booking_size|
+----------+---------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+------------+----------+------------+
|     B1023|     C005|  River of Dreams|  2026-10-12|2026-10-14|      5|             320|          10|     Confirmed|          Card|        1600|    1440.0|       Large|
|     B1029|     C005|Starlight Express|  2026-10-15|2026-10-17|      4|             320|           5|     Confirmed|          Card|        1280|    1216.0|       Large|
|     B1013|     C001|  The Last Island|  2026-10-07|2026-10-09|      5|             250|          10|     Confirmed|          Card|        1250|    1

**C. Aggregations and joins**

14. Count bookings for each booking_status.

In [0]:
clean_df.groupBy("booking_status").count().show()

+--------------+-----+
|booking_status|count|
+--------------+-----+
|      Refunded|    3|
|     Confirmed|   25|
|     Cancelled|    4|
+--------------+-----+



15. For Confirmed bookings only, calculate total tickets and net revenue for each movie_title.

In [0]:
clean_df.filter(F.col("booking_status") == "Confirmed").groupBy("movie_title").agg(F.sum("tickets").alias("total_tickets"),
         F.round(F.sum("net_amount"), 2).alias("net_revenue")).orderBy(F.col("net_revenue").desc()).show()


+-----------------+-------------+-----------+
|      movie_title|total_tickets|net_revenue|
+-----------------+-------------+-----------+
|  The Last Island|           26|     5905.0|
|Starlight Express|           23|     5500.0|
|         Code Red|           13|     3087.0|
|  River of Dreams|           10|     2380.0|
+-----------------+-------------+-----------+



16. Find the top 3 cinema_id values by Confirmed net revenue.

In [0]:
clean_df.filter(F.col("booking_status") == "Confirmed").groupBy("cinema_id").agg(F.round(F.sum("net_amount"), 2).alias("net_revenue")).orderBy(F.col("net_revenue").desc()).limit(3).show()


+---------+-----------+
|cinema_id|net_revenue|
+---------+-----------+
|     C005|     3840.0|
|     C001|     3075.0|
|     C002|     2610.0|
+---------+-----------+



17. Left join the bookings DataFrame with the cinema master on cinema_id.

In [0]:
joined_df = clean_df.join(cinemas_df, on="cinema_id", how="left")
joined_df.show()

+---------+----------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+------------+----------+------------+--------------------+--------------------+------------------+------------+
|cinema_id|booking_id|      movie_title|booking_date| show_date|tickets|price_per_ticket|discount_pct|booking_status|payment_method|gross_amount|net_amount|booking_size|         cinema_name|          facilities|          location|screen_count|
+---------+----------+-----------------+------------+----------+-------+----------------+------------+--------------+--------------+------------+----------+------------+--------------------+--------------------+------------------+------------+
|     C001|     B1001|Starlight Express|  2026-10-01|2026-10-03|      2|             250|          10|     Confirmed|           UPI|         500|     450.0|     Regular|      Galaxy Screens|[Parking, Food Co...| {Hyderabad, West}|           5|
|     C002|     B1002|  

18. From the joined DataFrame, show cinema_name, location.city, movie_title and booking_status.

In [0]:
joined_df.select("cinema_name", F.col("location.city").alias("city"),"movie_title", "booking_status").show()

+--------------------+---------+-----------------+--------------+
|         cinema_name|     city|      movie_title|booking_status|
+--------------------+---------+-----------------+--------------+
|      Galaxy Screens|Hyderabad|Starlight Express|     Confirmed|
|Silverlight Multi...|Bengaluru|  The Last Island|     Confirmed|
|        Metro Movies|   Mumbai|         Code Red|     Cancelled|
|      Coastal Cinema|  Chennai|  River of Dreams|     Confirmed|
|    Capital Cineplex|    Delhi|Starlight Express|     Confirmed|
|      Orchid Screens|     Pune|  The Last Island|     Confirmed|
|      Galaxy Screens|Hyderabad|         Code Red|     Confirmed|
|Silverlight Multi...|Bengaluru|  River of Dreams|      Refunded|
|        Metro Movies|   Mumbai|Starlight Express|     Confirmed|
|      Coastal Cinema|  Chennai|  The Last Island|     Confirmed|
|    Capital Cineplex|    Delhi|         Code Red|     Confirmed|
|      Orchid Screens|     Pune|  River of Dreams|     Cancelled|
|      Gal

19. Calculate Confirmed net revenue by city. Identify bookings with no matching cinema master record.

In [0]:
joined_df.filter(F.col("booking_status") == "Confirmed").groupBy(F.col("location.city").alias("city")).agg(F.round(F.sum("net_amount"), 2).alias("net_revenue")).orderBy(F.col("net_revenue").desc()).show()

# Bookings with no matching cinema master record 
joined_df.filter(F.col("cinema_name").isNull()).show()

+---------+-----------+
|     city|net_revenue|
+---------+-----------+
|    Delhi|     3840.0|
|Hyderabad|     3075.0|
|Bengaluru|     2610.0|
|     Pune|     2520.0|
|   Mumbai|     2310.0|
|  Chennai|     2097.0|
|     NULL|      420.0|
+---------+-----------+

+---------+----------+---------------+------------+----------+-------+----------------+------------+--------------+--------------+------------+----------+------------+-----------+----------+--------+------------+
|cinema_id|booking_id|    movie_title|booking_date| show_date|tickets|price_per_ticket|discount_pct|booking_status|payment_method|gross_amount|net_amount|booking_size|cinema_name|facilities|location|screen_count|
+---------+----------+---------------+------------+----------+-------+----------------+------------+--------------+--------------+------------+----------+------------+-----------+----------+--------+------------+
|     C999|     B1030|The Last Island|  2026-10-15|2026-10-18|      2|             210|         

**D. JSON, dates and DataFrame quality**

20. Use the nested JSON field location.zone to display each cinema and its zone.

In [0]:
cinemas_df.select("cinema_id", "cinema_name", F.col("location.zone").alias("zone")).show()

+---------+--------------------+-------+
|cinema_id|         cinema_name|   zone|
+---------+--------------------+-------+
|     C001|      Galaxy Screens|   West|
|     C002|Silverlight Multi...|  North|
|     C003|        Metro Movies|Central|
|     C004|      Coastal Cinema|  South|
|     C005|    Capital Cineplex|   East|
|     C006|      Orchid Screens|   West|
+---------+--------------------+-------+



21. Explode the facilities array to produce one row per cinema and facility.

In [0]:
cinemas_df.select("cinema_id", "cinema_name", F.explode("facilities").alias("facility")).show()


+---------+--------------------+-----------------+
|cinema_id|         cinema_name|         facility|
+---------+--------------------+-----------------+
|     C001|      Galaxy Screens|          Parking|
|     C001|      Galaxy Screens|       Food Court|
|     C001|      Galaxy Screens|Wheelchair Access|
|     C002|Silverlight Multi...|          Parking|
|     C002|Silverlight Multi...|       Food Court|
|     C003|        Metro Movies|       Food Court|
|     C003|        Metro Movies|Wheelchair Access|
|     C004|      Coastal Cinema|          Parking|
|     C005|    Capital Cineplex|          Parking|
|     C005|    Capital Cineplex|       Food Court|
|     C005|    Capital Cineplex|Wheelchair Access|
|     C006|      Orchid Screens|       Food Court|
|     C006|      Orchid Screens|Wheelchair Access|
+---------+--------------------+-----------------+



22. Convert booking_date and show_date into date columns and calculate
days_between_booking_and_show.

In [0]:
dates_df = (
    clean_df
    .withColumn("booking_date", F.to_date("booking_date"))
    .withColumn("show_date", F.to_date("show_date"))
    .withColumn("days_between_booking_and_show", F.datediff("show_date", "booking_date"))
)
dates_df.printSchema()
dates_df.select("booking_id", "booking_date", "show_date", "days_between_booking_and_show").show()

root
 |-- booking_id: string (nullable = true)
 |-- cinema_id: string (nullable = true)
 |-- movie_title: string (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- show_date: date (nullable = true)
 |-- tickets: integer (nullable = true)
 |-- price_per_ticket: integer (nullable = true)
 |-- discount_pct: integer (nullable = false)
 |-- booking_status: string (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- gross_amount: integer (nullable = true)
 |-- net_amount: double (nullable = true)
 |-- booking_size: string (nullable = false)
 |-- days_between_booking_and_show: integer (nullable = true)

+----------+------------+----------+-----------------------------+
|booking_id|booking_date| show_date|days_between_booking_and_show|
+----------+------------+----------+-----------------------------+
|     B1001|  2026-10-01|2026-10-03|                            2|
|     B1002|  2026-10-01|2026-10-02|                            1|
|     B1003|  2026-10-02|2026-

23. Identify repeated booking_id values and remove duplicate bookings while retaining one row per
booking_id.

In [0]:
dupes = dates_df.groupBy("booking_id").count().filter(F.col("count") > 1)
display(dupes)
display(dates_df.join(dupes.select("booking_id"), "booking_id"))   
 
dedup_df = dates_df.dropDuplicates(["booking_id"])
print("Before:", dates_df.count(), " After:", dedup_df.count())

booking_id,count
B1006,2
B1021,2


booking_id,cinema_id,movie_title,booking_date,show_date,tickets,price_per_ticket,discount_pct,booking_status,payment_method,gross_amount,net_amount,booking_size,days_between_booking_and_show
B1006,C006,The Last Island,2026-10-03,2026-10-04,5,210,0,Confirmed,UPI,1050,1050.0,Large,1
B1021,C003,The Last Island,2026-10-11,2026-10-13,2,220,0,Confirmed,UPI,440,440.0,Regular,2
B1006,C006,The Last Island,2026-10-03,2026-10-04,5,210,0,Confirmed,UPI,1050,1050.0,Large,1
B1021,C003,The Last Island,2026-10-11,2026-10-13,2,220,0,Confirmed,UPI,440,440.0,Regular,2


Before: 32  After: 30


**E. Window functions and Delta**

24. Use a window function to rank movies by Confirmed net revenue (highest first).

In [0]:
movie_rev = (
    dedup_df.filter(F.col("booking_status") == "Confirmed")
    .groupBy("movie_title")
    .agg(F.round(F.sum("net_amount"), 2).alias("net_revenue"))
)
w = Window.orderBy(F.col("net_revenue").desc())
movie_rev.withColumn("revenue_rank", F.dense_rank().over(w)).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------------+-----------+------------+
|      movie_title|net_revenue|revenue_rank|
+-----------------+-----------+------------+
|Starlight Express|     5500.0|           1|
|  The Last Island|     4415.0|           2|
|         Code Red|     3087.0|           3|
|  River of Dreams|     2380.0|           4|
+-----------------+-----------+------------+



25. Save the cleaned, deduplicated, joined DataFrame as a Delta table and query five rows with SQL.

In [0]:
final_df = dedup_df.join(cinemas_df, on="cinema_id", how="left")
 
table_name = f"{catalog}.{schema}.cinema_bookings_clean"
final_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
 
display(spark.sql(f"SELECT * FROM {table_name} LIMIT 5"))

cinema_id,booking_id,movie_title,booking_date,show_date,tickets,price_per_ticket,discount_pct,booking_status,payment_method,gross_amount,net_amount,booking_size,days_between_booking_and_show,cinema_name,facilities,location,screen_count
C001,B1001,Starlight Express,2026-10-01,2026-10-03,2,250,10,Confirmed,UPI,500,450.0,Regular,2,Galaxy Screens,"List(Parking, Food Court, Wheelchair Access)","List(Hyderabad, West)",5
C001,B1007,Code Red,2026-10-04,2026-10-06,3,250,0,Confirmed,UPI,750,750.0,Regular,2,Galaxy Screens,"List(Parking, Food Court, Wheelchair Access)","List(Hyderabad, West)",5
C005,B1017,The Last Island,2026-10-09,2026-10-11,4,320,15,Refunded,Card,1280,1088.0,Large,2,Capital Cineplex,"List(Parking, Food Court, Wheelchair Access)","List(Delhi, East)",8
C006,B1018,Starlight Express,2026-10-09,2026-10-11,2,210,0,Confirmed,UPI,420,420.0,Regular,2,Orchid Screens,"List(Food Court, Wheelchair Access)","List(Pune, West)",4
C002,B1020,Code Red,2026-10-10,2026-10-13,3,300,10,Confirmed,Card,900,810.0,Regular,3,Silverlight Multiplex,"List(Parking, Food Court)","List(Bengaluru, North)",7


**Final challenge**

Build a clean cinema performance report: deduplicate bookings, fill missing values, calculate discounted
revenue for Confirmed bookings, join cinema details, aggregate by city and movie, and save the result as
a Delta table.

In [0]:
report_df = (
    final_df.filter(F.col("booking_status") == "Confirmed")
    .groupBy(F.col("location.city").alias("city"), "movie_title")
    .agg(F.sum("tickets").alias("total_tickets"),
         F.round(F.sum("gross_amount"), 2).alias("gross_revenue"),
         F.round(F.sum("net_amount"), 2).alias("discounted_revenue"),
         F.countDistinct("booking_id").alias("bookings"))
    .orderBy("city", F.col("discounted_revenue").desc())
)
 
report_table = f"{catalog}.{schema}.cinema_performance_report"
report_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(report_table)
display(spark.sql(f"SELECT * FROM {report_table}"))

city,movie_title,total_tickets,gross_revenue,discounted_revenue,bookings
null,The Last Island,2,420,420.0,1
Bengaluru,The Last Island,4,1200,1200.0,2
Bengaluru,Code Red,3,900,810.0,1
Bengaluru,Starlight Express,2,600,600.0,1
Chennai,River of Dreams,4,720,720.0,1
Chennai,Starlight Express,4,720,684.0,1
Chennai,Code Red,3,540,513.0,1
Chennai,The Last Island,1,180,180.0,1
Delhi,Starlight Express,6,1920,1760.0,2
Delhi,River of Dreams,5,1600,1440.0,1
